# 04 - Probabilistic Forecast Evaluation

This notebook evaluates ensemble monsoon onset forecasts using:
- **Fair Brier Skill Score (BSS)**: Accounts for ensemble size
- **Fair Ranked Probability Skill Score (RPSS)**: Multi-category
- **AUC**: Discrimination ability
- **Reliability Diagrams**: Calibration assessment

Reference: Decision-oriented benchmarking to transform AI weather forecast access:
Application to the Indian monsoon

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

# Import monsoon benchmark modules
from monsoon_benchmark.data.regions import CMZ_BOUNDS
from monsoon_benchmark.metrics import (
    compute_fair_brier_score,
    compute_brier_skill_score,
    compute_fair_rps,
    compute_rps_skill_score,
    compute_auc,
    compute_reliability_diagram,
)
from monsoon_benchmark.visualization import (
    plot_reliability_diagram,
    plot_roc_curve,
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Probabilistic Evaluation Setup

Configure 5-day bins for onset probability forecasts.

In [ ]:
# Probabilistic evaluation parameters
BIN_SIZE_DAYS = 5
N_BINS = 7  # 7 bins covering 35 days

# Bins: [1-5], [6-10], [11-15], [16-20], [21-25], [26-30], [31-35]
bin_edges = [(i*BIN_SIZE_DAYS + 1, (i+1)*BIN_SIZE_DAYS) for i in range(N_BINS)]

print("Onset probability bins (days after initialization):")
for i, (start, end) in enumerate(bin_edges):
    print(f"  Bin {i}: Days {start}-{end}")
print(f"  Bin {N_BINS}: No onset (after day 35)")

In [ ]:
# Ensemble models and their sizes
ENSEMBLE_MODELS = {
    'GenCast': 51,
    'AIFS-ENS': 51,
    'IFS-ENS': 51,
    'FuXi-S2S': 51,
    'Climatology': 124,  # One member per historical year
}

print("\nEnsemble models:")
for model, size in ENSEMBLE_MODELS.items():
    print(f"  {model}: {size} members")

## 2. Create Synthetic Ensemble Forecasts

For demonstration, create synthetic ensemble forecasts.

In [ ]:
np.random.seed(42)

# Number of forecast cases (init_date, grid_cell combinations)
N_CASES = 200

# Generate observed onset bins (ground truth)
# Most onsets occur in bins 2-4 (days 6-20)
observed_bins = np.random.choice(
    range(N_BINS + 1),  # 0 to 7 (including no-onset)
    size=N_CASES,
    p=[0.05, 0.15, 0.25, 0.25, 0.15, 0.10, 0.04, 0.01]  # Weighted toward middle bins
)

print(f"Generated {N_CASES} forecast cases")
print(f"Observed bin distribution:")
for i in range(N_BINS + 1):
    count = np.sum(observed_bins == i)
    pct = count / N_CASES * 100
    label = f"Days {bin_edges[i][0]}-{bin_edges[i][1]}" if i < N_BINS else "No onset"
    print(f"  Bin {i} ({label}): {count} ({pct:.1f}%)")

In [ ]:
def generate_ensemble_probs(observed_bins, ensemble_size, skill_level=0.7):
    """
    Generate synthetic ensemble probabilities.
    
    skill_level: 0=random, 1=perfect (higher = better skill)
    """
    n_cases = len(observed_bins)
    n_categories = N_BINS + 1
    
    probs = np.zeros((n_cases, n_categories))
    
    for i, obs_bin in enumerate(observed_bins):
        # Start with uniform probabilities
        base_probs = np.ones(n_categories) / n_categories
        
        # Add skill by concentrating probability on correct bin
        skill_probs = np.zeros(n_categories)
        skill_probs[obs_bin] = 1.0
        
        # Blend based on skill level
        probs[i] = (1 - skill_level) * base_probs + skill_level * skill_probs
        
        # Add noise and normalize
        noise = np.abs(np.random.normal(0, 0.1, n_categories))
        probs[i] = probs[i] + noise
        probs[i] = probs[i] / probs[i].sum()
    
    return probs

# Generate forecasts with different skill levels
ensemble_forecasts = {
    'GenCast': generate_ensemble_probs(observed_bins, 51, skill_level=0.65),
    'AIFS-ENS': generate_ensemble_probs(observed_bins, 51, skill_level=0.60),
    'IFS-ENS': generate_ensemble_probs(observed_bins, 51, skill_level=0.55),
    'FuXi-S2S': generate_ensemble_probs(observed_bins, 51, skill_level=0.50),
    'Climatology': generate_ensemble_probs(observed_bins, 124, skill_level=0.0),  # No skill
}

print("Generated ensemble forecasts")
print(f"Probability array shape: {ensemble_forecasts['GenCast'].shape}")

## 3. Compute Probabilistic Metrics

Evaluate using Fair Brier Score, Fair RPS, and AUC.

In [ ]:
# Convert observed bins to one-hot encoding
observed_onehot = np.zeros((N_CASES, N_BINS + 1))
for i, obs_bin in enumerate(observed_bins):
    observed_onehot[i, obs_bin] = 1

# Compute metrics for each model
probabilistic_results = {}

for model_name, probs in ensemble_forecasts.items():
    ensemble_size = ENSEMBLE_MODELS[model_name]
    
    # For binary Brier Score (onset in days 1-15 vs later/no onset)
    # Create 2D arrays for the binary case
    prob_onset_1_15 = probs[:, :3].sum(axis=1)  # P(onset in bins 0-2)
    prob_no_onset_1_15 = 1 - prob_onset_1_15    # P(no onset in bins 0-2)
    
    # Stack into 2D: (N_CASES, 2)
    binary_probs = np.stack([prob_no_onset_1_15, prob_onset_1_15], axis=1)
    
    # Binary observed: 1 if onset in days 1-15, 0 otherwise
    obs_onset_1_15 = (observed_bins < 3).astype(float)
    binary_obs = np.stack([1 - obs_onset_1_15, obs_onset_1_15], axis=1)
    
    # Fair Brier Score (binary)
    fair_bs = compute_fair_brier_score(binary_probs, binary_obs, ensemble_size)
    
    # Fair RPS (multi-category with all bins)
    fair_rps = compute_fair_rps(probs, observed_onehot, ensemble_size)
    
    # AUC for onset in days 1-15
    auc = compute_auc(prob_onset_1_15, obs_onset_1_15)
    
    probabilistic_results[model_name] = {
        'Fair BS': fair_bs,
        'Fair RPS': fair_rps,
        'AUC': auc,
        'probs': probs,
        'prob_onset_1_15': prob_onset_1_15,
        'ensemble_size': ensemble_size,
    }

print("Probabilistic Metrics:")
print(f"{'Model':<15} {'Fair BS':>10} {'Fair RPS':>10} {'AUC':>10}")
print("-" * 50)
for model_name, r in probabilistic_results.items():
    print(f"{model_name:<15} {r['Fair BS']:>10.4f} {r['Fair RPS']:>10.4f} {r['AUC']:>10.3f}")

In [ ]:
# Compute skill scores relative to climatology
clim_bs = probabilistic_results['Climatology']['Fair BS']
clim_rps = probabilistic_results['Climatology']['Fair RPS']

print("\nSkill Scores (% improvement over climatology):")
print(f"{'Model':<15} {'BSS (%)':>10} {'RPSS (%)':>10}")
print("-" * 40)

for model_name, r in probabilistic_results.items():
    if model_name == 'Climatology':
        continue
    
    bss = compute_brier_skill_score(r['Fair BS'], clim_bs) * 100
    rpss = compute_rps_skill_score(r['Fair RPS'], clim_rps) * 100
    
    probabilistic_results[model_name]['BSS'] = bss
    probabilistic_results[model_name]['RPSS'] = rpss
    
    print(f"{model_name:<15} {bss:>10.1f} {rpss:>10.1f}")

## 4. Reliability Diagrams

Assess forecast calibration with reliability diagrams.

In [ ]:
# Compute reliability data for each model
reliability_data = {}

for model_name, r in probabilistic_results.items():
    rel_bins = compute_reliability_diagram(
        r['prob_onset_1_15'],
        (observed_bins < 3).astype(float),
        n_bins=10
    )
    reliability_data[model_name] = rel_bins

print("Reliability data computed for all models")

In [ ]:
# Plot reliability diagrams
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

models_to_plot = ['GenCast', 'AIFS-ENS', 'IFS-ENS', 'FuXi-S2S', 'Climatology']

for ax, model_name in zip(axes, models_to_plot):
    rel_data = reliability_data[model_name]
    
    # Extract data
    forecast_probs = [b.forecast_prob_mean for b in rel_data]
    observed_freqs = [b.observed_frequency for b in rel_data]
    n_samples = [b.n_samples for b in rel_data]
    
    # Remove NaN
    valid = ~np.isnan(observed_freqs)
    forecast_probs = np.array(forecast_probs)[valid]
    observed_freqs = np.array(observed_freqs)[valid]
    
    # Perfect reliability line
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect', alpha=0.7)
    
    # Reliability curve
    ax.plot(forecast_probs, observed_freqs, 'o-', markersize=8, label=model_name)
    
    ax.set_xlabel('Forecast Probability')
    ax.set_ylabel('Observed Frequency')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.legend(loc='upper left')
    ax.set_title(f'{model_name}')
    ax.grid(True, alpha=0.3)

# Hide empty subplot
axes[-1].set_visible(False)

fig.suptitle('Reliability Diagrams - Onset in Days 1-15', fontsize=14)
plt.tight_layout()
plt.show()

## 5. ROC Curves

Assess discrimination with ROC curves and AUC.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Compute ROC curves
fig, ax = plt.subplots(figsize=(8, 8))

# No-skill line
ax.plot([0, 1], [0, 1], 'k--', label='No skill (AUC=0.5)')

colors = plt.cm.Set1(np.linspace(0, 1, len(models_to_plot)))

for model_name, color in zip(models_to_plot, colors):
    r = probabilistic_results[model_name]
    
    # Compute ROC curve
    obs_binary = (observed_bins < 3).astype(float)
    fpr, tpr, thresholds = roc_curve(obs_binary, r['prob_onset_1_15'])
    auc = r['AUC']
    
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{model_name} (AUC={auc:.3f})')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.legend(loc='lower right')
ax.set_title('ROC Curves - Onset in Days 1-15')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Probability Distribution by Bin

Examine how forecast probabilities are distributed across onset bins.

In [ ]:
# Mean forecast probabilities for each bin
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

bin_labels = [f"{bin_edges[i][0]}-{bin_edges[i][1]}d" for i in range(N_BINS)] + ['No onset']

for ax, model_name in zip(axes, models_to_plot):
    probs = probabilistic_results[model_name]['probs']
    mean_probs = probs.mean(axis=0)
    
    bars = ax.bar(range(N_BINS + 1), mean_probs, color='steelblue', alpha=0.7)
    ax.set_xticks(range(N_BINS + 1))
    ax.set_xticklabels(bin_labels, rotation=45, ha='right')
    ax.set_xlabel('Onset Bin')
    ax.set_ylabel('Mean Probability')
    ax.set_title(f'{model_name}')
    ax.set_ylim(0, 0.5)
    ax.grid(True, alpha=0.3, axis='y')

axes[-1].set_visible(False)

fig.suptitle('Mean Forecast Probability by Onset Bin', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Model Ranking Summary

In [ ]:
# Create summary DataFrame
summary_data = []
for model_name in models_to_plot:
    r = probabilistic_results[model_name]
    summary_data.append({
        'Model': model_name,
        'Ensemble Size': r['ensemble_size'],
        'Fair BS': r['Fair BS'],
        'Fair RPS': r['Fair RPS'],
        'AUC': r['AUC'],
        'BSS (%)': r.get('BSS', 0),
        'RPSS (%)': r.get('RPSS', 0),
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('BSS (%)', ascending=False)

print("\nProbabilistic Forecast Ranking:")
print(summary_df.to_string(index=False))

In [ ]:
# Bar chart of skill scores
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models_sorted = summary_df[summary_df['Model'] != 'Climatology']['Model'].values
colors = plt.cm.Set2(np.linspace(0, 1, len(models_sorted)))

# BSS
ax1 = axes[0]
bss_vals = summary_df[summary_df['Model'].isin(models_sorted)]['BSS (%)'].values
ax1.barh(models_sorted, bss_vals, color=colors)
ax1.axvline(0, color='k', linestyle='-', linewidth=0.5)
ax1.set_xlabel('BSS (%)')
ax1.set_title('Brier Skill Score')
ax1.grid(True, alpha=0.3, axis='x')

# RPSS
ax2 = axes[1]
rpss_vals = summary_df[summary_df['Model'].isin(models_sorted)]['RPSS (%)'].values
ax2.barh(models_sorted, rpss_vals, color=colors)
ax2.axvline(0, color='k', linestyle='-', linewidth=0.5)
ax2.set_xlabel('RPSS (%)')
ax2.set_title('Ranked Probability Skill Score')
ax2.grid(True, alpha=0.3, axis='x')

# AUC
ax3 = axes[2]
auc_vals = summary_df[summary_df['Model'].isin(models_sorted)]['AUC'].values
ax3.barh(models_sorted, auc_vals, color=colors)
ax3.axvline(0.5, color='k', linestyle='--', label='No skill')
ax3.set_xlabel('AUC')
ax3.set_title('Area Under ROC Curve')
ax3.set_xlim(0.4, 1.0)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("=" * 60)
print("PROBABILISTIC EVALUATION SUMMARY")
print("=" * 60)

best_model = summary_df.iloc[0]['Model']
best_bss = summary_df.iloc[0]['BSS (%)']
best_rpss = summary_df.iloc[0]['RPSS (%)']
best_auc = summary_df.iloc[0]['AUC']

print(f"\n1. Best performing model: {best_model}")
print(f"   - BSS: {best_bss:.1f}%")
print(f"   - RPSS: {best_rpss:.1f}%")
print(f"   - AUC: {best_auc:.3f}")

print(f"\n2. Key findings:")
print(f"   - All AIWP models show positive skill over climatology")
print(f"   - GenCast has highest AUC (best discrimination)")
print(f"   - Reliability varies across models")

print(f"\n3. Evaluation parameters:")
print(f"   - 5-day onset bins")
print(f"   - Fair metrics adjusted for ensemble size")
print(f"   - Binary event: onset in days 1-15")

## Next Steps

Continue to:
- **05_2025_case_study.ipynb**: Apply benchmark to 2025 monsoon season